# Modulo 9 · Lezione 4 — Addestramento ML

**Progetto Finale: Sistema Intelligente di Predizione ed Esposizione di Modelli**

---

## Cosa vedremo oggi?

| # | Argomento |
|---|------|
| 1 | Preparare i dati per Scikit-Learn |
| 2 | Baseline + modelli a confronto |
| 3 | Pipeline completa (preprocessing + modello) |
| 4 | GridSearchCV — ricerca degli iperparametri |
| 5 | Salvare e caricare il modello con joblib |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
import joblib
import json

Path('models').mkdir(exist_ok=True)

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
print(f"Dataset caricato: {df.shape}")
df.head(3)

---
## 1. Preparare i dati per Scikit-Learn

In [ ]:
FEATURES = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol',
    'so2_ratio', 'acid_index', 'density_diff', 'tipo_encoded'
]
TARGET = 'buen_vino'  # classificazione binaria: vino buono (quality >= 7)

X = df[FEATURES].values
y = df[TARGET].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Classe 1 (buono): {y.mean():.2%}  |  Classe 0 (normale): {(1-y).mean():.2%}")

# Split stratificato (conserva la proporzione delle classi)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]} righe | Test: {X_test.shape[0]} righe")
print(f"Classe 1 in train: {y_train.mean():.2%}  |  in test: {y_test.mean():.2%}")

---
## 2. Baseline + confronto tra modelli

In [ ]:
# Baseline: predice sempre la classe maggioritaria
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_f1 = f1_score(y_test, baseline.predict(X_test))
print(f"Baseline F1: {baseline_f1:.4f} (sempre classe maggioritaria)")

# Confronto tra modelli SENZA ottimizzazione (per vedere il punto di partenza)
modelos = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

resultados = {}
print(f"\n{'Modello':25} {'F1':>8} {'AUC':>8} {'Acc':>8}")
print('-' * 53)
for nombre, modelo in modelos.items():
    modelo.fit(X_train_sc, y_train)
    y_pred = modelo.predict(X_test_sc)
    y_prob = modelo.predict_proba(X_test_sc)[:, 1]
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    acc = (y_pred == y_test).mean()
    resultados[nombre] = {'f1': f1, 'auc': auc, 'acc': acc}
    print(f"  {nombre:23} {f1:8.4f} {auc:8.4f} {acc:8.4f}")

In [ ]:
# Curve ROC
fig, ax = plt.subplots(figsize=(8, 6))

for nombre, modelo in modelos.items():
    y_prob = modelo.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = resultados[nombre]['auc']
    ax.plot(fpr, tpr, lw=2, label=f"{nombre} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Baseline')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Curve ROC — confronto tra modelli')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Pipeline Scikit-Learn

Una `Pipeline` concatena passaggi di preprocessing + modello. Vantaggi:
- Evita il **data leakage**: lo scaler impara solo dai dati di addestramento
- Permette di usare `cross_val_score` e `GridSearchCV` in modo corretto
- L'oggetto salvato con `joblib` contiene tutto: preprocessing + modello

In [ ]:
# Pipeline: StandardScaler + RandomForest
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Cross-validation (5 fold stratificati)
cv_scores = cross_val_score(
    pipe, X_train, y_train,
    cv=5, scoring='f1', n_jobs=-1
)
print(f"CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Score per fold: {cv_scores.round(4)}")

# Addestrare su tutto il train
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print(f"\nTest F1:  {f1_score(y_test, y_pred):.4f}")
print(f"Test AUC: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Bueno']))

In [ ]:
# Matrice di confusione
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                fontsize=16, color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred Normal', 'Pred Bueno'])
ax.set_yticklabels(['Real Normal', 'Real Bueno'])
ax.set_title('Matrice di confusione — Random Forest')
plt.tight_layout()
plt.show()

---
## 4. GridSearchCV — ricerca degli iperparametri

In [ ]:
# Nota sulla notazione: in una Pipeline, i parametri si accedono
# con il formato 'nome_step__parametro'

param_grid = {
    'modelo__n_estimators':  [100, 200],
    'modelo__max_depth':     [None, 10, 20],
    'modelo__min_samples_split': [2, 5],
    'modelo__class_weight':  [None, 'balanced'],
}

print(f"Combinazioni da esplorare: {2 * 3 * 2 * 2} × 5 fold = {2*3*2*2*5} fit")

grid_search = GridSearchCV(
    pipe, param_grid,
    cv=5, scoring='f1',
    n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nMigliori iperparametri:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nCV F1 del miglior modello: {grid_search.best_score_:.4f}")

# Valutare il miglior modello sul test
mejor_modelo = grid_search.best_estimator_
y_pred_opt   = mejor_modelo.predict(X_test)
y_prob_opt   = mejor_modelo.predict_proba(X_test)[:, 1]

print(f"\nTest F1 (ottimizzato):  {f1_score(y_test, y_pred_opt):.4f}")
print(f"Test AUC (ottimizzato): {roc_auc_score(y_test, y_prob_opt):.4f}")

In [ ]:
# Importanza delle feature
rf = mejor_modelo.named_steps['modelo']
importancias = pd.Series(rf.feature_importances_, index=FEATURES)
importancias = importancias.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
importancias.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_ylabel('Importanza (mean decrease impurity)')
ax.set_title('Feature Importance — Random Forest ottimizzato')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Top 5 feature più importanti:")
for feat, imp in importancias.head(5).items():
    print(f"  {feat:30}: {imp:.4f}")

---
## 5. Salvare e caricare il modello con joblib

In [ ]:
from datetime import datetime

MODEL_PATH = 'models/rf_vinos_v1.joblib'
META_PATH  = 'models/rf_vinos_v1_meta.json'

# Salvare la Pipeline completa (scaler + modello)
joblib.dump(mejor_modelo, MODEL_PATH)
print(f"✓ Modello salvato: {MODEL_PATH}")
print(f"  Dimensione: {Path(MODEL_PATH).stat().st_size / 1024:.0f} KB")

# Metadati del modello
meta = {
    'nombre':        'RandomForest Wine Quality Classifier',
    'version':       'v1',
    'fecha_entrenamiento': datetime.now().isoformat(),
    'features':      FEATURES,
    'target':        TARGET,
    'umbral_target': 7,
    'n_train':       int(X_train.shape[0]),
    'n_test':        int(X_test.shape[0]),
    'metricas_test': {
        'f1':       round(float(f1_score(y_test, y_pred_opt)), 4),
        'auc':      round(float(roc_auc_score(y_test, y_prob_opt)), 4),
        'accuracy': round(float((y_pred_opt == y_test).mean()), 4)
    },
    'mejores_hiperparametros': {k: str(v) for k, v in grid_search.best_params_.items()},
    'archivo_modelo': MODEL_PATH
}

with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"✓ Metadati salvati: {META_PATH}")

In [ ]:
# Verificare che il modello caricato dia gli stessi risultati
modelo_cargado = joblib.load(MODEL_PATH)

# Predire un nuovo esempio
ejemplo = np.array([[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0,
                     0.9978, 3.51, 0.56, 9.4,
                     11/34, 7.4-0.7, abs(0.9978-1.0), 1]])  # rosso

pred = modelo_cargado.predict(ejemplo)[0]
prob = modelo_cargado.predict_proba(ejemplo)[0, 1]

print(f"Predizione per l'esempio:")
print(f"  Classe: {'BUONO' if pred == 1 else 'NORMALE'} (buen_vino={pred})")
print(f"  Probabilità di essere buono: {prob:.4f}")
print(f"\nModello pronto per essere servito dall'API (Lezione 6)")

---
## Pratica

### Esercizio 1 — Regressione: predire il punteggio esatto

**Consegna:** Oltre a classificare (buono/normale), vogliamo predire il punteggio esatto (0-10).

1. Usa `quality` come target (regressione, non classificazione)
2. Prova `RandomForestRegressor` e `GradientBoostingRegressor`
3. Valuta con MAE e RMSE
4. Quale modello commette meno errore?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
FEATURES = ['fixed acidity','volatile acidity','citric acid','residual sugar',
            'chlorides','free sulfur dioxide','total sulfur dioxide','density',
            'pH','sulphates','alcohol','so2_ratio','acid_index','density_diff','tipo_encoded']

X = df[FEATURES].values
y = df['quality'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

for nombre, reg in [
    ('Random Forest',    RandomForestRegressor(n_estimators=100, random_state=42)),
    ('Gradient Boost',  GradientBoostingRegressor(n_estimators=100, random_state=42)),
]:
    pipe = Pipeline([('scaler', StandardScaler()), ('modelo', reg)])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = root_mean_squared_error(y_te, pred)
    print(f"{nombre:20} MAE={mae:.4f}  RMSE={rmse:.4f}")
```
</details>

---
### Esercizio 2 — Soglia di classificazione

**Consegna:** Per impostazione predefinita, `predict()` usa la soglia 0.5. Ma possiamo modificarla.

1. Ottieni `y_prob = modelo.predict_proba(X_test)[:, 1]`
2. Prova soglie da 0.3 a 0.7 (passi di 0.05)
3. Per ogni soglia calcola Precision, Recall e F1
4. Quale soglia massimizza F1? Ha senso dal punto di vista del business?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# Assumiamo mejor_modelo e X_test, y_test dalla lezione
y_prob = mejor_modelo.predict_proba(X_test)[:, 1]

umbrales = np.arange(0.3, 0.71, 0.05)
datos = []
for u in umbrales:
    pred = (y_prob >= u).astype(int)
    datos.append({
        'umbral':    round(u, 2),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall':    recall_score(y_test, pred, zero_division=0),
        'f1':        f1_score(y_test, pred, zero_division=0),
    })

df_u = pd.DataFrame(datos)
mejor = df_u.loc[df_u['f1'].idxmax()]

print(df_u.round(4).to_string(index=False))
print(f"\nSoglia migliore: {mejor['umbral']} (F1={mejor['f1']:.4f})")
print("Business: una soglia bassa (0.3) evita di perdere buoni vini (alto recall)")
print("ma accetta più falsi positivi. Dipende dal costo dell'errore.")

fig, ax = plt.subplots(figsize=(9, 4))
for col in ['precision', 'recall', 'f1']:
    ax.plot(df_u['umbral'], df_u[col], marker='o', label=col)
ax.axvline(mejor['umbral'], ls='--', color='red', label=f"soglia ottimale ({mejor['umbral']})")
ax.set_xlabel('Soglia'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 3 — Validazione incrociata stratificata

**Consegna:** Confronta tre modelli usando `StratifiedKFold` con 10 fold:

1. `LogisticRegression`, `RandomForestClassifier`, `GradientBoostingClassifier`
2. Metrica: AUC (`scoring='roc_auc'`)
3. Disegna un boxplot con i 10 score di ogni modello
4. Quale è più coerente (varianza minore)?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
FEATURES = ['fixed acidity','volatile acidity','citric acid','residual sugar',
            'chlorides','free sulfur dioxide','total sulfur dioxide','density',
            'pH','sulphates','alcohol','so2_ratio','acid_index','density_diff','tipo_encoded']
X = df[FEATURES].values; y = df['buen_vino'].values

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_todos = {}

for nombre, clf in [
    ('Logistic', LogisticRegression(max_iter=1000, random_state=42)),
    ('RF',       RandomForestClassifier(100, random_state=42)),
    ('GB',       GradientBoostingClassifier(100, random_state=42)),
]:
    pipe = Pipeline([('sc', StandardScaler()), ('m', clf)])
    sc = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    scores_todos[nombre] = sc
    print(f"{nombre:12}: AUC {sc.mean():.4f} ± {sc.std():.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(scores_todos.values(), labels=scores_todos.keys(), patch_artist=True)
ax.set_ylabel('AUC'); ax.set_title('CV-10 AUC per modello')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 4 — Funzione `predecir_vino()`

**Consegna:** Crea una funzione di predizione pronta per l'API:

```python
def predecir_vino(datos: dict) -> dict:
    """
    Input: dict con i valori di un vino
    Output: {'prediccion': 'bueno'/'normal', 'probabilidad': 0.82, 'version': 'v1'}
    """
```

La funzione deve:
1. Caricare il modello da `models/rf_vinos_v1.joblib`
2. Calcolare internamente `so2_ratio`, `acid_index`, `density_diff`
3. Gestire il caso in cui manchi qualche campo (raise ValueError con messaggio chiaro)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import joblib
import numpy as np

CAMPOS_REQUERIDOS = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
    'density', 'pH', 'sulphates', 'alcohol', 'tipo'
]
FEATURES_ORDEN = [
    'fixed acidity','volatile acidity','citric acid','residual sugar',
    'chlorides','free sulfur dioxide','total sulfur dioxide','density',
    'pH','sulphates','alcohol','so2_ratio','acid_index','density_diff','tipo_encoded'
]

_modelo_cache = None

def predecir_vino(datos: dict) -> dict:
    global _modelo_cache

    faltantes = [c for c in CAMPOS_REQUERIDOS if c not in datos]
    if faltantes:
        raise ValueError(f"Campi mancanti: {faltantes}")

    if _modelo_cache is None:
        _modelo_cache = joblib.load('models/rf_vinos_v1.joblib')

    d = dict(datos)
    d['so2_ratio']    = d['free sulfur dioxide'] / (d['total sulfur dioxide'] + 1e-8)
    d['acid_index']   = d['fixed acidity'] - d['volatile acidity']
    d['density_diff'] = abs(d['density'] - 1.0)
    d['tipo_encoded'] = 1 if d['tipo'] == 'tinto' else 0

    X = np.array([[d[f] for f in FEATURES_ORDEN]])
    pred = int(_modelo_cache.predict(X)[0])
    prob = float(_modelo_cache.predict_proba(X)[0, 1])

    return {
        'prediccion':  'bueno' if pred == 1 else 'normal',
        'probabilidad': round(prob, 4),
        'version': 'v1'
    }

# Test
resultado = predecir_vino({
    'fixed acidity': 7.4, 'volatile acidity': 0.7, 'citric acid': 0.0,
    'residual sugar': 1.9, 'chlorides': 0.076, 'free sulfur dioxide': 11.0,
    'total sulfur dioxide': 34.0, 'density': 0.9978, 'pH': 3.51,
    'sulphates': 0.56, 'alcohol': 9.4, 'tipo': 'tinto'
})
print(resultado)
```
</details>

---
## Riepilogo

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score
import joblib

# Preparare
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Pipeline
pipe = Pipeline([('sc', StandardScaler()), ('m', RandomForestClassifier(random_state=42))])

# Ricerca degli iperparametri
gs = GridSearchCV(pipe, {'m__n_estimators': [100, 200], 'm__max_depth': [None, 10]},
                  cv=5, scoring='f1', n_jobs=-1)
gs.fit(X_tr, y_tr)

# Valutare
y_pred = gs.best_estimator_.predict(X_te)
print(f1_score(y_te, y_pred), roc_auc_score(y_te, gs.best_estimator_.predict_proba(X_te)[:,1]))

# Salvare
joblib.dump(gs.best_estimator_, 'models/modelo.joblib')
```

## Cosa vedremo nella prossima lezione?
Nella **Lezione 5** addestreremo una rete neurale con **TensorFlow/Keras** sugli stessi dati sui vini, confronteremo le sue prestazioni con il Random Forest e salveremo il modello in formato `.keras`.